# Modelo Prescritivo

## Funções dos Modelos

- **Modelo preditivo** → estima probabilidades  
- **Modelo prescritivo** → toma decisão ótima com base nessas probabilidades  

## Perguntas que cada UM responde

- **Preditivo**: “Qual a chance?”  
- **Prescritivo**: “O que eu devo fazer?”  

## Ferramenta utilizada

Vamos utilizar  **Otimização Linear Inteira Binária** - 
- Usa equações lineares;
- Busca maximizar ou minimizar algo;
- Só permite decisões 0 ou 1.


In [82]:
# import pulp

# videos = ['A','B','C']
# probabilidades = [0.72,0.65,0.40]

# Integração entre Modelo Preditivo e Modelo Prescritivo

## Objetivo

O objetivo desta etapa foi integrar o modelo preditivo (Random Forest) ao modelo prescritivo (Otimização com PuLP), mantendo uma arquitetura modular e organizada.

---

## Estrutura do Projeto

O projeto foi dividido em dois módulos:

- `predictive_model.py` → Responsável por treinar o modelo e gerar probabilidades.
- `prescriptive_model.ipynb` → Responsável por utilizar essas probabilidades para tomar a melhor decisão via otimização matemática.

Essa separação permite reutilização de código e melhor organização do sistema.

---

## Como a Integração Foi Feita

Para importar o modelo preditivo dentro do notebook prescritivo, foi necessário adicionar dinamicamente o diretório ao caminho de execução do Python:

```python
import sys
import os

# Adiciona a pasta do modelo preditivo ao path do Python
sys.path.append(os.path.abspath("../predictive-model"))

from predictive_model import (
    recomendar_top_n,
    modelo,
    le_titulo,
    le_proximo,
    le_periodo
)

In [91]:
# Importações
import sys
import os
# adiciona a pasta predictive-model no path
sys.path.append(os.path.abspath("../predictive-model"))

from predictive_model import (
    recomendar_top_n,
    modelo,
    le_titulo,
    le_proximo,
    le_periodo
)

print(type(modelo))
print(type(le_titulo))
print(type(le_proximo))
print(type(le_periodo))
print(type(recomendar_top_n))


<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'sklearn.preprocessing._label.LabelEncoder'>
<class 'sklearn.preprocessing._label.LabelEncoder'>
<class 'sklearn.preprocessing._label.LabelEncoder'>
<class 'function'>


In [84]:
#Problema 
prob=pulp.LpProblem("Recomendacao",pulp.LpMaximize)
print(prob)


Recomendacao:
MAXIMIZE
None
VARIABLES



# TESTE

In [85]:
# Testar com dado do dataSeat
print(le_titulo.classes_[:5])

['https://music.youtube.com/watch?v=-3gkan9wSaQ'
 'https://music.youtube.com/watch?v=-KrC-gqKTMg'
 'https://music.youtube.com/watch?v=-Uw97CvOgKU'
 'https://music.youtube.com/watch?v=-ZBMFq4-gQ4'
 'https://music.youtube.com/watch?v=-aytZ0n_KNQ']


In [86]:
video_teste = le_titulo.classes_[0]
periodo_teste = "noite"

video_teste = le_titulo.classes_[0]
periodo_teste = "noite"

resultado = recomendar_top_n(
    video_teste,
    periodo_teste,
    modelo,
    le_titulo,
    le_proximo,
    le_periodo,
    n=3
)

print("\n RECOMENDAÇÕES GERADAS")
print(f"Vídeo atual: {video_teste}")
print(f"Período: {periodo_teste}")
print("-" * 50)

if resultado:
    for i, (link, proba) in enumerate(resultado, start=1):
        print(f"{i}º lugar")
        print(f" Link: {link}")
        print(f" Probabilidade estimada: {proba:.2%}")
        print("-" * 50)
else:
    print("Nenhuma recomendação encontrada.")


 RECOMENDAÇÕES GERADAS
Vídeo atual: https://music.youtube.com/watch?v=-3gkan9wSaQ
Período: noite
--------------------------------------------------
1º lugar
 Link: https://music.youtube.com/watch?v=CNDYWba7YDY
 Probabilidade estimada: 45.00%
--------------------------------------------------
2º lugar
 Link: https://music.youtube.com/watch?v=V3NbrjVPIHM
 Probabilidade estimada: 18.00%
--------------------------------------------------
3º lugar
 Link: https://music.youtube.com/watch?v=KjteCvkh0B8
 Probabilidade estimada: 16.00%
--------------------------------------------------


C:\Users\leticiasilva-ieg\AppData\Roaming\Python\Python311\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [87]:
# Váriaveis Binárias
x=pulp.LpVariable.dicts("video",videos,cat='Binary')
print(x)


{'A': video_A, 'B': video_B, 'C': video_C}


In [88]:
prob += pulp.lpSum([probabilidades[i] * x[videos[i]] for i in range(len(videos))])
print(prob)

Recomendacao:
MAXIMIZE
0.72*video_A + 0.65*video_B + 0.4*video_C + 0.0
VARIABLES
0 <= video_A <= 1 Integer
0 <= video_B <= 1 Integer
0 <= video_C <= 1 Integer



In [89]:
# Restrições - Vamos escolher apenas um video 
prob+=pulp.lpSum([x[v]for v in videos])==1
print(prob)


Recomendacao:
MAXIMIZE
0.72*video_A + 0.65*video_B + 0.4*video_C + 0.0
SUBJECT TO
_C1: video_A + video_B + video_C = 1

VARIABLES
0 <= video_A <= 1 Integer
0 <= video_B <= 1 Integer
0 <= video_C <= 1 Integer



In [90]:
prob.solve()

for v in videos:
    if x[v].value() == 1:
        print("Recomendar:", v)

Recomendar: A
